In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import TRAIN_SEASONS
from src.features.build import add_elo
from src.evaluation.metrics import log_loss_hda

m = pd.read_csv("../data/processed/matches.csv", parse_dates=["date"])
m["season"] = m["season"].astype(str).str.zfill(4)
print(len(m), TRAIN_SEASONS)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from src.evaluation.metrics import to_hda
from src.models.baseline import FrequencyBaseline

def eval_elo(k, home_adv, n_splits=5):
    """Log loss fuera de muestra (solo entrenamiento) de un Elo con estos parámetros."""
    e = add_elo(m, k=k, home_adv=home_adv)      # el Elo se calcula con todo el historial, de forma secuencial
    tr = (e[e["season"].isin(TRAIN_SEASONS)]
            .sort_values(["date", "match_id"]).reset_index(drop=True))
    X = tr[["elo_diff"]].to_numpy() / 100        # escala más estable para la logística
    y = tr["result"].to_numpy()

    p_elo = np.full((len(tr), 3), np.nan)
    p_base = np.full((len(tr), 3), np.nan)
    for a, b in TimeSeriesSplit(n_splits=n_splits).split(X):
        clf = LogisticRegression().fit(X[a], y[a])
        p_elo[b] = to_hda(clf, clf.predict_proba(X[b]))     # reordena a H, D, A
        p_base[b] = FrequencyBaseline().fit(None, y[a]).predict_proba(X[b])

    ok = ~np.isnan(p_elo[:, 0])                  # solo filas que cayeron en algún bloque de prueba
    return log_loss_hda(y[ok], p_elo[ok]), log_loss_hda(y[ok], p_base[ok])

ll_elo, ll_base = eval_elo(k=20, home_adv=60)    # valores actuales de config.py
print(f"Baseline de frecuencias: {ll_base:.4f}")
print(f"Elo (K=20, ventaja=60):  {ll_elo:.4f}")
print(f"Mejora:                  {ll_base - ll_elo:.4f}")

In [ ]:
resultados = []
for k in [10, 15, 20, 30, 40]:
    for adv in [30, 50, 60, 80, 100]:
        ll, _ = eval_elo(k, adv)
        resultados.append({"K": k, "ventaja": adv, "log_loss": ll})
res = pd.DataFrame(resultados)

tabla = res.pivot(index="K", columns="ventaja", values="log_loss")
print(tabla.round(4))
mejor = res.loc[res["log_loss"].idxmin()]
print(f"\nMejor: K={mejor.K:.0f}, ventaja={mejor.ventaja:.0f} → {mejor.log_loss:.4f}")
print(f"Actual (20, 60): {tabla.loc[20, 60]:.4f}")

import seaborn as sns
sns.heatmap(tabla, annot=True, fmt=".4f", cmap="viridis_r")
plt.title("Log loss (menor = mejor)")
plt.tight_layout()
plt.savefig("../docs/elo_grid.png", dpi=150)
plt.show()

In [ ]:
def add_elo_v2(df, k=30, home_adv=60, init=1500.0, regress=0.0, promoted_init=1500.0):
    ratings, cur_season, first_season = {}, None, df["season"].min()
    home_pre, away_pre = {}, {}
    for r in df.sort_values(["date", "match_id"]).itertuples():
        if r.season != cur_season:
            if cur_season is not None:                      # empieza una temporada nueva
                for t in ratings:
                    ratings[t] = init + (1 - regress) * (ratings[t] - init)
            cur_season = r.season
        start = init if r.season == first_season else promoted_init
        rh, ra = ratings.get(r.home, start), ratings.get(r.away, start)
        home_pre[r.match_id], away_pre[r.match_id] = rh, ra
        exp_h = 1 / (1 + 10 ** ((ra - (rh + home_adv)) / 400))
        delta = k * ({"H": 1.0, "D": 0.5, "A": 0.0}[r.result] - exp_h)
        ratings[r.home], ratings[r.away] = rh + delta, ra - delta
    out = df.copy()
    out["home_elo"] = out["match_id"].map(home_pre)
    out["away_elo"] = out["match_id"].map(away_pre)
    out["elo_diff"] = out["home_elo"] + home_adv - out["away_elo"]
    return out

def eval_from(e, n_splits=5):
    tr = (e[e["season"].isin(TRAIN_SEASONS)]
            .sort_values(["date", "match_id"]).reset_index(drop=True))
    X, y = tr[["elo_diff"]].to_numpy() / 100, tr["result"].to_numpy()
    p = np.full((len(tr), 3), np.nan)
    for a, b in TimeSeriesSplit(n_splits=n_splits).split(X):
        clf = LogisticRegression().fit(X[a], y[a])
        p[b] = to_hda(clf, clf.predict_proba(X[b]))
    ok = ~np.isnan(p[:, 0])
    return log_loss_hda(y[ok], p[ok])

# Comprobación: sin regresión ni ascendidos debe reproducir el Elo anterior (K=30 → 0.9781)
print("v2 neutro:", round(eval_from(add_elo_v2(m, k=30, home_adv=60)), 4))

filas = []
for reg in [0.0, 0.1, 0.2, 0.3, 0.5]:
    for prom in [1500, 1450, 1400, 1350]:
        filas.append({"regress": reg, "ascendidos": prom,
                      "log_loss": eval_from(add_elo_v2(m, regress=reg, promoted_init=prom))})
g = pd.DataFrame(filas).pivot(index="regress", columns="ascendidos", values="log_loss")
print(g.round(4))